# Trening segmentacji temporalnej (3 klatki → maska środkowa)

Wariant `train_long.ipynb`: wejście to **3 kolejne klatki RGB** z tej samej sekwencji, wyjście to segmentacja **środkowej** klatki. Uruchamiaj komórki po kolei od góry.

In [ ]:
import glob
import os
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset

import pytorch_lightning as pl
from pytorch_lightning import Trainer
import torchvision.models as models
import torchvision.transforms as T

CONFIG = {
    "data_root": "../CARLA_processed",
    "max_images_per_sequence": 300,
    "batch_size": 2,
    "num_workers": 2,
    "train_sequences": ["00", "01", "02", "03"],
    "val_sequences": ["04"],
    "test_sequences": ["05"],
    "epochs": 10,
    "lr": 1e-4,
    "loss_ignore_index": 0,
    "checkpoint_path": "tempo_segmentation_3frame.ckpt",
    "seed": 42,
}

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])

In [7]:
def build_label_lut(class_map: dict, size: int = 256) -> np.ndarray:
    lut = np.zeros(size, dtype=np.int64)
    for raw, idx in class_map.items():
        lut[int(raw)] = int(idx)
    return lut


NUM_CLASSES = 13
class_map = {i: i for i in range(NUM_CLASSES)}
LABEL_LUT = build_label_lut(class_map)

In [8]:
class CARLATripletDataset(Dataset):
    """3 kolejne klatki RGB → maska segmentacji klatki środkowej."""

    def __init__(
        self,
        root: str,
        label_lut: np.ndarray,
        max_per_sequence: int | None = None,
        sequences: list[str] | None = None,
    ):
        self.samples: list[tuple[str, str, str, str]] = []
        self.label_lut = label_lut

        all_folders = sorted(
            f for f in os.listdir(root)
            if os.path.isdir(os.path.join(root, f)) and f.isdigit()
        )
        if sequences is not None:
            seq_set = set(sequences)
            all_folders = [f for f in all_folders if f in seq_set]

        for folder in all_folders:
            rgb_dir = os.path.join(root, folder, "rgb")
            mask_dir = os.path.join(root, folder, "segmentation_raw")
            if not os.path.isdir(rgb_dir) or not os.path.isdir(mask_dir):
                continue

            img_paths = sorted(glob.glob(os.path.join(rgb_dir, "*.png")))
            if max_per_sequence is not None:
                img_paths = img_paths[:max_per_sequence]

            for i in range(1, len(img_paths) - 1):
                p_prev, p_mid, p_next = img_paths[i - 1], img_paths[i], img_paths[i + 1]
                mask_path = os.path.join(mask_dir, os.path.basename(p_mid))
                if os.path.exists(mask_path):
                    self.samples.append((p_prev, p_mid, p_next, mask_path))

        self.img_tf = T.Compose([T.ToTensor()])

    def __len__(self):
        return len(self.samples)

    def _load_rgb(self, path: str) -> torch.Tensor:
        return self.img_tf(Image.open(path).convert("RGB"))

    def _load_mask(self, path: str) -> torch.Tensor:
        mask = np.array(Image.open(path))[:, :, 0]
        mask = self.label_lut[mask.astype(np.int64)]
        return torch.from_numpy(mask).long()

    def __getitem__(self, idx):
        p_prev, p_mid, p_next, mask_path = self.samples[idx]
        frames = torch.stack(
            [self._load_rgb(p) for p in (p_prev, p_mid, p_next)],
            dim=0,
        )  # (3, C, H, W)
        mask = self._load_mask(mask_path)  # (H, W) — klatka środkowa
        return frames, mask

In [9]:
def _make_resnet_encoder():
    resnet = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
    return nn.Sequential(
        resnet.conv1,
        resnet.bn1,
        resnet.relu,
        resnet.maxpool,
        resnet.layer1,
        resnet.layer2,
        resnet.layer3,
        resnet.layer4,
    )


def _make_decoder(num_classes: int) -> nn.Sequential:
    return nn.Sequential(
        nn.ConvTranspose2d(512, 256, 2, stride=2),
        nn.ReLU(),
        nn.ConvTranspose2d(256, 128, 2, stride=2),
        nn.ReLU(),
        nn.ConvTranspose2d(128, 64, 2, stride=2),
        nn.ReLU(),
        nn.ConvTranspose2d(64, 32, 2, stride=2),
        nn.ReLU(),
        nn.ConvTranspose2d(32, 16, 2, stride=2),
        nn.ReLU(),
        nn.Conv2d(16, num_classes, 1),
    )


class TempoSegModel(nn.Module):
    """Współdzielony enkoder ResNet34 na każdej z 3 klatek, fuzja cech, dekoder."""

    def __init__(self, num_classes: int, n_frames: int = 3):
        super().__init__()
        self.n_frames = n_frames
        self.encoder = _make_resnet_encoder()
        self.fuse = nn.Sequential(
            nn.Conv2d(512 * n_frames, 512, kernel_size=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
        )
        self.decoder = _make_decoder(num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, C, H, W)
        feats = [self.encoder(x[:, t]) for t in range(self.n_frames)]
        fused = self.fuse(torch.cat(feats, dim=1))
        return self.decoder(fused)


class LitTempoSegModel(pl.LightningModule):
    def __init__(self, model: nn.Module, lr: float = 1e-4, ignore_index: int = 0):
        super().__init__()
        self.model = model
        self.lr = lr
        self.ignore_index = ignore_index
        self.criterion = nn.CrossEntropyLoss(ignore_index=ignore_index)

    def forward(self, x):
        return self.model(x)

    def _step(self, batch):
        frames, masks = batch  # frames (B, 3, C, H, W), masks (B, H, W)
        preds = self(frames)
        preds = F.interpolate(
            preds,
            size=masks.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        return self.criterion(preds, masks)

    def training_step(self, batch, batch_idx):
        loss = self._step(batch)
        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss = self._step(batch)
        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)

    def configure_optimizers(self):
        trainable = (p for p in self.model.parameters() if p.requires_grad)
        return torch.optim.Adam(trainable, lr=self.lr)

In [10]:
root = CONFIG["data_root"]
_ds_kw = dict(
    label_lut=LABEL_LUT,
    max_per_sequence=CONFIG["max_images_per_sequence"],
)

train_ds = CARLATripletDataset(root, sequences=CONFIG["train_sequences"], **_ds_kw)
val_ds = CARLATripletDataset(root, sequences=CONFIG["val_sequences"], **_ds_kw)
test_ds = CARLATripletDataset(root, sequences=CONFIG["test_sequences"], **_ds_kw)

train_n, val_n, test_n = len(train_ds), len(val_ds), len(test_ds)
n = train_n + val_n + test_n
if n == 0:
    raise RuntimeError(
        "Brak próbek — sprawdź data_root i strukturę (np. 00/rgb, 00/segmentation_raw). "
        "Potrzeba co najmniej 3 klatek na sekwencję."
    )

train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=torch.cuda.is_available(),
    persistent_workers=CONFIG["num_workers"] > 0,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=torch.cuda.is_available(),
    persistent_workers=CONFIG["num_workers"] > 0,
)
test_loader = DataLoader(
    test_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=torch.cuda.is_available(),
    persistent_workers=CONFIG["num_workers"] > 0,
)

print(
    f"Triplet dataset: {n} próbek (train={train_n}, val={val_n}, test={test_n}) "
    f"→ {len(train_loader)} / {len(val_loader)} / {len(test_loader)} batchy"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch_model = TempoSegModel(num_classes=NUM_CLASSES).to(device)
lit_model = LitTempoSegModel(
    torch_model,
    lr=CONFIG["lr"],
    ignore_index=CONFIG["loss_ignore_index"],
)

accelerator = "gpu" if torch.cuda.is_available() else "cpu"
trainer = Trainer(max_epochs=CONFIG["epochs"], accelerator=accelerator, devices=1)

print(f"Start treningu na {device} ({CONFIG['epochs']} epok)…")
trainer.fit(lit_model, train_loader, val_loader)
trainer.save_checkpoint(CONFIG["checkpoint_path"])
print("Zapisano:", CONFIG["checkpoint_path"])

Triplet dataset: 1788 próbek (train=1192, val=298, test=298) → 298 / 75 / 75 batchy


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type             | Params | Mode  | FLOPs
---------------------------------------------------------------
0 | model     | TempoSegModel    | 22.8 M | train | 0    
1 | criterion | CrossEntropyLoss | 0      | train | 0    
---------------------------------------------------------------
22.8 M    Trainable params
0         Non-trainable params
22.8 M    Total params
91.087    Total estimated model params size (MB)
132       

Start treningu na cuda (10 epok)…


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 3.81 GiB of which 9.56 MiB is free. Process 3712 has 614.00 MiB memory in use. Process 5363 has 882.00 MiB memory in use. Including non-PyTorch memory, this process has 2.32 GiB memory in use. Of the allocated memory 2.13 GiB is allocated by PyTorch, and 100.78 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import matplotlib.pyplot as plt

_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seg_demo = TempoSegModel(num_classes=NUM_CLASSES).to(_device)
_ckpt = torch.load(CONFIG["checkpoint_path"], map_location=_device)
_sd = {
    k.replace("model.", "", 1): v
    for k, v in _ckpt["state_dict"].items()
    if k.startswith("model.")
}
seg_demo.load_state_dict(_sd, strict=True)
seg_demo.eval()

try:
    _test_ds = test_loader.dataset
except NameError:
    _test_ds = CARLATripletDataset(
        CONFIG["data_root"],
        LABEL_LUT,
        max_per_sequence=CONFIG["max_images_per_sequence"],
        sequences=CONFIG["test_sequences"],
    )


def _idx_to_rgb(mask_hw, n_classes):
    lut = (plt.cm.turbo(np.linspace(0, 0.92, n_classes))[:, :3]).astype(np.float32)
    m = np.clip(mask_hw.detach().cpu().numpy(), 0, n_classes - 1)
    return lut[m]


N_SHOW = 10
indices = random.sample(range(len(_test_ds)), min(N_SHOW, len(_test_ds)))

with torch.no_grad():
    for n_i, idx in enumerate(indices, start=1):
        _frames, _mask = _test_ds[idx]
        _frames = _frames.unsqueeze(0).to(_device)
        _mask = _mask.unsqueeze(0)
        _logits = seg_demo(_frames)
        _logits = F.interpolate(
            _logits,
            size=_mask.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        _pred = _logits.argmax(dim=1)
        _rgb = _frames[0, 1].cpu().permute(1, 2, 0).numpy()
        _rgb = np.clip(_rgb, 0.0, 1.0)
        _fig, _ax = plt.subplots(1, 3, figsize=(14, 4))
        _ax[0].imshow(_rgb)
        _ax[0].set_title("Wejście — klatka środkowa (RGB)")
        _ax[0].axis("off")
        _ax[1].imshow(_idx_to_rgb(_pred[0], NUM_CLASSES))
        _ax[1].set_title("Predykcja (argmax)")
        _ax[1].axis("off")
        _ax[2].imshow(_idx_to_rgb(_mask[0], NUM_CLASSES))
        _ax[2].set_title("Ground truth (środek)")
        _ax[2].axis("off")
        _fig.suptitle(f"Przykład {n_i} / {len(indices)} — zbiór testowy (3 klatki → środek)")
        plt.tight_layout()
        plt.show()